# Avaliação causal do Programa Mais Médicos

**Atividade - Fundamentos da inferência causal**

Este notebook aplica o quadro de resultados potenciais e o critério de backdoor a uma política pública federal diferente do Bolsa Família. O estudo de caso é o **Programa Mais Médicos (PMM)** e o desfecho é a taxa municipal de internações por condições sensíveis à atenção primária (ICSAP).

> **Nota metodológica:** a parte empírica usa dados simulados exclusivamente para demonstrar os conceitos. Portanto, os resultados numéricos não constituem uma avaliação real do PMM.

**Objetivos da atividade**

1. formular uma pergunta causal e definir os resultados potenciais;
2. escolher e interpretar o estimando de interesse;
3. propor um DAG com confundidores, mediador e colisor;
4. aplicar o critério de backdoor;
5. justificar o que deve e o que não deve ser controlado;
6. ilustrar, por simulação, as consequências das escolhas de ajuste.


## 1. Política, pergunta causal e unidade de análise

O Programa Mais Médicos amplia a disponibilidade de profissionais em localidades com dificuldades de provimento. Uma cadeia causal plausível é: recebimento de médicos $\rightarrow$ maior acesso à atenção básica $\rightarrow$ prevenção e manejo precoce de doenças $\rightarrow$ redução de internações evitáveis.

**Pergunta causal:** qual é o efeito do recebimento de profissionais do PMM sobre a taxa de ICSAP dos municípios que receberam o programa, no período posterior à implementação?

- **Unidade de análise:** município.
- **População-alvo:** municípios brasileiros elegíveis no período definido pelo estudo.
- **Tratamento $D_i$:** indicador igual a 1 se o município $i$ recebeu profissionais do PMM no início do período e 0 caso contrário.
- **Resultado $Y_i$:** taxa de ICSAP por 10 mil habitantes, medida após o tratamento.
- **Horizonte:** por exemplo, média dos dois anos posteriores à entrada. Em uma aplicação real, ano de entrada, janela e regras de elegibilidade devem ser fixados antes da análise.


## 2. Quadro de resultados potenciais

Para cada município $i$, definimos:

$$Y_i(1)=\text{taxa de ICSAP que ocorreria se o município recebesse o PMM}$$

$$Y_i(0)=\text{taxa de ICSAP que ocorreria se o município não recebesse o PMM}$$

O efeito causal individual seria $\tau_i=Y_i(1)-Y_i(0)$, mas somente um dos dois resultados potenciais é observado:

$$Y_i=D_iY_i(1)+(1-D_i)Y_i(0).$$

Esse é o problema fundamental da inferência causal: não observamos o contrafactual da mesma unidade. A avaliação precisa construir uma comparação que represente adequadamente o resultado não observado.

### Estimando escolhido: ATT

$$ATT=E[Y(1)-Y(0)\mid D=1].$$

O ATT responde: **qual foi o efeito médio do PMM nos municípios que efetivamente receberam a política?** Ele é adequado porque a política já foi implementada de modo seletivo e o interesse principal é reconstruir o que teria ocorrido com os municípios atendidos na ausência do programa. O ATE seria mais adequado para uma decisão sobre universalização; o ATU, para estimar o possível efeito nos municípios ainda não atendidos.

A comparação bruta $E[Y\mid D=1]-E[Y\mid D=0]$ não identifica necessariamente o ATT, pois a seleção dos municípios não é aleatória.

## 3. DAG proposto

O DAG registra hipóteses causais, não meras correlações. Os nós são:

- $V$: vulnerabilidade socioeconômica prévia;
- $B$: oferta/cobertura prévia de atenção básica;
- $G$: capacidade administrativa municipal prévia;
- $D$: recebimento do PMM;
- $M$: acesso/consultas de atenção básica após o PMM (**mediador**);
- $U$: choque epidemiológico não observado;
- $K$: gasto adicional em saúde após o tratamento (**colisor plausível**);
- $Y$: taxa posterior de ICSAP.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

plt.style.use('seaborn-v0_8-whitegrid')
RANDOM_SEED = 20260723

def desenhar_dag():
    pos = {
        'V': (0.05, 0.78), 'B': (0.05, 0.50), 'G': (0.05, 0.22),
        'D': (0.38, 0.50), 'M': (0.65, 0.72), 'K': (0.65, 0.25),
        'U': (0.38, 0.05), 'Y': (0.93, 0.50)
    }
    labels = {
        'V': 'Vulnerabilidade\nprévia', 'B': 'Atenção básica\nprévia',
        'G': 'Capacidade\nadministrativa', 'D': 'Recebimento\ndo PMM',
        'M': 'Acesso/consultas\npós-PMM', 'K': 'Gasto adicional\npós-PMM',
        'U': 'Choque\nepidemiológico', 'Y': 'Internações\nevitáveis'
    }
    edges = [
        ('V', 'D'), ('V', 'Y'), ('B', 'D'), ('B', 'Y'),
        ('G', 'D'), ('G', 'Y'), ('D', 'M'), ('M', 'Y'),
        ('D', 'Y'), ('D', 'K'), ('U', 'K'), ('U', 'Y')
    ]
    colors = {'M': '#f4a261', 'K': '#e76f51', 'D': '#457b9d', 'Y': '#2a9d8f'}
    fig, ax = plt.subplots(figsize=(12, 6))
    for a, b in edges:
        ax.annotate('', xy=pos[b], xytext=pos[a],
                    arrowprops=dict(arrowstyle='-|>', lw=1.6, color='#555',
                                    shrinkA=28, shrinkB=28))
    for node, (x, y) in pos.items():
        ax.text(x, y, labels[node], ha='center', va='center', fontsize=10,
                color='white' if node in colors else '#222',
                bbox=dict(boxstyle='round,pad=0.5', fc=colors.get(node, '#dbe7f0'), ec='#345'))
    ax.set(xlim=(-0.08, 1.06), ylim=(-0.08, 0.93), title='DAG causal proposto')
    ax.axis('off')
    plt.show()

desenhar_dag()


## 4. Critério de backdoor e conjunto de ajuste

Os caminhos que entram em $D$ por uma seta e produzem confundimento são:

$$D\leftarrow V\rightarrow Y,$$
$$D\leftarrow B\rightarrow Y,$$
$$D\leftarrow G\rightarrow Y.$$

O conjunto pré-tratamento $Z=\{V,B,G\}$ bloqueia esses caminhos e não contém descendentes de $D$. Assim, ele satisfaz o critério de backdoor **se o DAG estiver corretamente especificado**. Sob ignorabilidade condicional,

$$(Y(1),Y(0))\perp D\mid Z,$$

o efeito pode ser identificado por padronização. Para o ATT:

$$ATT=E\{E[Y\mid D=1,Z]-E[Y\mid D=0,Z]\mid D=1\}.$$

### O que controlar

| Variável | Papel | Controlar para o efeito total? | Justificativa |
|---|---|---:|---|
| Vulnerabilidade prévia ($V$) | Confundidor | Sim | Causa a priorização e as internações |
| Atenção básica prévia ($B$) | Confundidor | Sim | Afeta o recebimento e o desfecho |
| Capacidade administrativa prévia ($G$) | Confundidor | Sim | Afeta adesão/execução e resultados |
| Acesso/consultas pós-PMM ($M$) | Mediador | Não | Ajustar bloquearia parte do efeito total |
| Gasto adicional pós-PMM ($K$) | Colisor | Não | Ajustar abre $D\rightarrow K\leftarrow U\rightarrow Y$ |
| Choque epidemiológico ($U$) | Causa do resultado | Não observado | Não confunde $D$ no DAG original; torna-se associado a $D$ se condicionarmos em $K$ |

O mediador somente entraria em uma análise especificamente voltada ao efeito direto, com hipóteses adicionais. O colisor não deve ser condicionado nem usado para selecionar a amostra.

## 5. Simulação do problema causal

A simulação abaixo torna observáveis os dois resultados potenciais apenas para fins pedagógicos. Na realidade, somente `Y` estaria disponível. O processo gerador foi construído para conter os três confundidores, o mediador e o colisor representados no DAG.


In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
n = 8_000

# Variáveis pré-tratamento e choque não observado
V = rng.normal(size=n)  # maior valor = maior vulnerabilidade
B = rng.normal(size=n)  # maior valor = melhor oferta prévia
G = rng.normal(size=n)  # maior valor = maior capacidade administrativa
U = rng.normal(size=n)  # choque epidemiológico

# Seleção não aleatória para o tratamento
logit_p = -0.3 + 1.1*V - 0.9*B + 0.6*G
p = 1 / (1 + np.exp(-logit_p))
D = rng.binomial(1, p)

# Resultados potenciais do mediador: o PMM aumenta o acesso
eps_m = rng.normal(scale=0.7, size=n)
M0 = 0.4 - 0.4*V + 0.8*B + 0.4*G + eps_m
M1 = M0 + 1.2

# Mesma perturbação em Y(0) e Y(1) para explicitar o efeito individual
eps_y = rng.normal(scale=1.2, size=n)
base = 18 + 2.0*V - 1.4*B - 0.5*G + 2.0*U + eps_y
Y0 = base - 1.8*M0
Y1 = base - 0.8 - 1.8*M1  # efeito direto -0,8 + efeito mediado -2,16
Y = D*Y1 + (1-D)*Y0

# Colisor: consequência conjunta de D e do choque U
K = 1.2*D + 1.3*U + rng.normal(scale=0.7, size=n)

df = pd.DataFrame({'D': D, 'Y': Y, 'Y0': Y0, 'Y1': Y1,
                   'V': V, 'B': B, 'G': G, 'M': D*M1 + (1-D)*M0,
                   'K': K, 'U': U, 'p': p})

df.head()


In [ ]:
# Diagnóstico de seleção: médias pré-tratamento por grupo
balanceamento = df.groupby('D')[['V', 'B', 'G']].mean().T
balanceamento.columns = ['Não tratado', 'Tratado']
balanceamento['Diferença'] = balanceamento['Tratado'] - balanceamento['Não tratado']
balanceamento.round(3)


Diferenças prévias entre tratados e não tratados mostram por que a comparação bruta não é um contrafactual adequado. Balanceamento é um diagnóstico descritivo; por si só, não prova ausência de confundimento.

In [ ]:
def diferenca_bruta(data):
    return data.loc[data.D == 1, 'Y'].mean() - data.loc[data.D == 0, 'Y'].mean()

def g_computation_att(data, covariaveis):
    tratados = data['D'].eq(1)
    modelos = {}
    pred = {}
    for d in [0, 1]:
        amostra = data['D'].eq(d)
        modelo = LinearRegression().fit(data.loc[amostra, covariaveis], data.loc[amostra, 'Y'])
        modelos[d] = modelo
        pred[d] = modelo.predict(data.loc[tratados, covariaveis])
    return np.mean(pred[1] - pred[0])

def coeficiente_tratamento(data, covariaveis):
    X = data[['D'] + covariaveis]
    return LinearRegression().fit(X, data['Y']).coef_[0]

att_verdadeiro = (df.loc[df.D == 1, 'Y1'] - df.loc[df.D == 1, 'Y0']).mean()
estimativas = pd.Series({
    'ATT verdadeiro (simulação)': att_verdadeiro,
    'Comparação bruta': diferenca_bruta(df),
    'Backdoor: g-computation com V, B e G': g_computation_att(df, ['V', 'B', 'G']),
    'Regressão com confundidores': coeficiente_tratamento(df, ['V', 'B', 'G']),
    'Sobreajuste: inclui mediador M': coeficiente_tratamento(df, ['V', 'B', 'G', 'M']),
    'Viés de colisor: inclui K': coeficiente_tratamento(df, ['V', 'B', 'G', 'K'])
})
resultado = estimativas.to_frame('Estimativa').assign(
    Viés=lambda x: x['Estimativa'] - att_verdadeiro
)
resultado.round(3)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
cores = ['#264653', '#e76f51', '#2a9d8f', '#2a9d8f', '#f4a261', '#9b5de5']
resultado['Estimativa'].plot.barh(ax=ax, color=cores)
ax.axvline(att_verdadeiro, color='black', linestyle='--', label='ATT verdadeiro')
ax.set(title='Como a escolha do ajuste altera a estimativa', xlabel='Efeito na taxa de ICSAP')
ax.legend()
plt.tight_layout()
plt.show()


## 6. Interpretação dos resultados simulados

1. **ATT verdadeiro:** pode ser calculado apenas porque a simulação revela $Y(1)$ e $Y(0)$. O valor esperado é $-2{,}96$: efeito direto de $-0{,}8$ somado ao efeito mediado de $-1{,}8\times1{,}2=-2{,}16$.
2. **Comparação bruta:** mistura o efeito do PMM com diferenças prévias entre municípios.
3. **Ajuste backdoor:** ao condicionar em $V$, $B$ e $G$, a g-computation reconstrói o resultado contrafactual dos tratados sob não tratamento e se aproxima do ATT verdadeiro.
4. **Controle do mediador:** incluir $M$ remove o caminho $D\rightarrow M\rightarrow Y$; a estimativa passa a se aproximar do efeito direto, não do efeito total solicitado.
5. **Controle do colisor:** incluir $K$ abre $D\rightarrow K\leftarrow U\rightarrow Y$ e induz associação entre $D$ e o choque epidemiológico $U$, gerando viés.

Esses resultados dependem do processo gerador especificado. Em dados reais, não conhecemos o DAG verdadeiro nem o ATT, razão pela qual teoria institucional, conhecimento substantivo e análises de sensibilidade são indispensáveis.

## 7. Hipóteses de identificação e limitações

- **Consistência:** “receber o PMM” precisa representar uma intervenção bem definida. Diferenças de dose, duração e perfil dos médicos desafiam essa hipótese.
- **Não interferência (SUTVA):** o tratamento de um município não deveria alterar o resultado de outro. Deslocamento de pacientes e médicos entre municípios pode violá-la.
- **Positividade:** para cada perfil de $V$, $B$ e $G$, deve haver probabilidade positiva de tratamento e não tratamento. Prioridades rígidas podem produzir falta de sobreposição.
- **Ausência de confundimento não observado:** após ajustar $Z$, não deve restar causa comum relevante de $D$ e $Y$. Essa hipótese não pode ser confirmada somente pelos dados.
- **Ordenação temporal:** confundidores devem ser medidos antes do PMM; medidas posteriores podem ser descendentes do tratamento.
- **Mensuração:** erros na identificação da entrada no programa, nas ICSAP e nas covariáveis podem introduzir viés.
- **Validade externa:** o ATT dos municípios tratados não é automaticamente transportável aos não tratados.

Em uma avaliação aplicada, seriam necessários dados administrativos do Ministério da Saúde/DATASUS, definição documental das regras de priorização, inspeção de sobreposição, erros-padrão compatíveis com a estrutura dos dados e análises de robustez. Se houver painel municipal e adoção escalonada, métodos longitudinais podem complementar a estratégia, mas não substituem a explicitação do estimando e das hipóteses causais.

## 8. Conclusão

A estratégia proposta identifica o **efeito total médio do PMM sobre os municípios tratados (ATT)** mediante ajuste por vulnerabilidade socioeconômica, atenção básica prévia e capacidade administrativa prévia. Essas variáveis bloqueiam os caminhos de backdoor postulados. O acesso posterior à atenção básica não deve ser controlado porque medeia parte do efeito; o gasto adicional posterior não deve ser controlado porque funciona como colisor no DAG proposto.

A conclusão causal é condicional: ela depende da adequação do DAG, da consistência, da positividade, da ausência de interferência e, principalmente, da inexistência de confundimento residual relevante após o ajuste.

## Referências bibliográficas

- ANGRIST, Joshua D.; PISCHKE, Jörn-Steffen. *Mastering 'Metrics: The Path from Cause to Effect*. Princeton University Press, 2015. Ver especialmente o capítulo 1, sobre resultados potenciais, viés de seleção e experimentos.
- BRASIL. Casa Civil da Presidência da República et al. *Avaliação de políticas públicas: guia prático de análise ex post*. v. 2. Brasília: Casa Civil da Presidência da República, 2018. Ver o capítulo 9, sobre causalidade e avaliação de impacto.
- FACURE, Matheus. *Causal Inference for the Brave and True*. Capítulos 1 e 2. Material indicado na disciplina. Ver a introdução à inferência causal, notação de resultados potenciais e grafos causais.
- MENEZES FILHO, Naercio (org.). *Avaliação econômica de projetos sociais*. 3. ed. São Paulo: Fundação Itaú Social, 2017. Ver o capítulo 2, “Modelo de Resultados Potenciais”, especialmente pp. 39-49.
- PEARL, Judea; MACKENZIE, Dana. *The Book of Why: The New Science of Cause and Effect*. New York: Basic Books, 2018. Ver os capítulos 1, 4 e 8, sobre grafos causais, confundimento, ajuste backdoor e contrafactuais.

### Declaração de uso da simulação

O código e os dados deste notebook foram produzidos para fins didáticos. Para uma submissão acadêmica, recomenda-se executar todas as células, conferir se as saídas foram preservadas e adaptar capa, identificação do aluno e padrão de citações às orientações formais da disciplina.